# 14 November submission drafts — Task 3 then Task 5

Slot 1 is the previously prepared Task 3 log-target candidate, based on scored v9. Slots 2–5 keep **Tasks 1–4 exactly equal to slot 1** and change only Task 5 using four prespecified hypotheses. This notebook prepares local CSVs and lineage cards; it does not upload files or use leaderboard feedback. The Task 5 detector scores come from already fitted competition-only models.

In [1]:
from pathlib import Path
import sys,json,hashlib,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
SEED=set_seed()
import numpy as np,pandas as pd
from scipy.stats import rankdata
SRC=ROOT/'src';OUT=SRC/'submission';TASK5=SRC/'task5'
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
slot1_path=OUT/'kaggle_t12345_t3_log_v13.csv'
v9_path=OUT/'kaggle_t12345_t5_alert25_v9.csv'
v10_path=OUT/'kaggle_t12345_t5_alert35_v10.csv'
v11_path=OUT/'kaggle_t12345_t5_gmm_match_v9_v11.csv'
paths=[slot1_path,v9_path,v10_path,v11_path]
for p in paths:
    assert p.exists() and p.with_suffix('.json').exists(),p
base_card=json.loads(slot1_path.with_suffix('.json').read_text(encoding='utf-8'))
assert base_card['output_sha256']==sha(slot1_path)
v9_card=json.loads(v9_path.with_suffix('.json').read_text(encoding='utf-8'))
v10_card=json.loads(v10_path.with_suffix('.json').read_text(encoding='utf-8'))
v11_card=json.loads(v11_path.with_suffix('.json').read_text(encoding='utf-8'))
assert v9_card['sha256']==sha(v9_path)
assert v10_card['sha256']==sha(v10_path)
assert v11_card['sha256']==sha(v11_path)
frames=[pd.read_csv(p,dtype=str) for p in paths]
slot1,v9,v10,v11=frames
columns=['id','task1','task2','task3','task4','task5']
sample=pd.read_csv(OUT/'sample_submission.csv',usecols=['id'],dtype=str)
for frame in frames:
    assert frame.columns.tolist()==columns and len(frame)==len(sample)==25889
    assert frame.id.is_unique and frame.id.tolist()==sample.id.tolist()
assert slot1[['id','task1','task2','task4','task5']].equals(v9[['id','task1','task2','task4','task5']])
assert slot1.task3.ne(v9.task3).any()
for frame in [v10,v11]:
    assert frame[['id','task1','task2','task3','task4']].equals(v9[['id','task1','task2','task3','task4']])
print('Slot 1 hash:',sha(slot1_path),'rows:',len(slot1))

Slot 1 hash: a1a0260892735af3e4a04142f6ca768b12e99fe978c3cf139feba53040006bbe rows: 25889


## 1. Reconstruct fitted Task 5 ranking and verify row order

Task 5 Test has player IDs, while the final sample has `ANS` IDs. The existing assembly maps them by unchanged row position. Verify that the saved 25% and GMM detector wrappers reproduce the v9/v11 binary columns exactly before using their raw scores.

In [2]:
data_card=json.loads((TASK5/'data'/'eda'/'eda_1_v2.json').read_text(encoding='utf-8'))
assert set(data_card['splits'])=={'test'}
test=pd.read_parquet(SRC/data_card['splits']['test']['path'])
assert len(test)==len(slot1) and test.player_id.is_unique and 'is_anomaly' not in test
X=test.drop(columns='player_id')
def saved_detector(name):
    model_card=json.loads((TASK5/'save'/f'{name}.json').read_text(encoding='utf-8'))
    assert model_card['data_name']=='eda_1_v2'
    model=joblib.load(SRC/model_card['model_path'])
    return model,model_card
ocsvm25,_=saved_detector('t5_r4_ocsvm_nu10_alert25_v4')
ocsvm35,_=saved_detector('t5_r5_ocsvm_nu10_alert35_v5')
gmm25,_=saved_detector('t5_r6_gmm_match_v9_alerts_v6')
o25=v9.task5.astype(int).to_numpy();o35=v10.task5.astype(int).to_numpy();g25=v11.task5.astype(int).to_numpy()
assert np.array_equal(ocsvm25.predict(X),o25)
assert np.array_equal(ocsvm35.predict(X),o35)
assert np.array_equal(gmm25.predict(X),g25)
assert o25.sum()==g25.sum()==6531 and o35.sum()==9118
assert np.array_equal(slot1.task5.astype(int),o25)
print('Verified exact detector predictions and Test row order')

Verified exact detector predictions and Test row order


## 2. Prespecified Task 5 hypotheses

- **Slot 2 — budget:** OCSVM 35% tests whether broader recall helps beyond v9's 25% alert level.
- **Slot 3 — consensus:** alert only when both OCSVM and GMM alert, testing whether agreement improves precision enough to offset fewer alerts.
- **Slot 4 — union:** alert when either model alerts, testing whether complementary coverage improves recall enough to offset extra alerts.
- **Slot 5 — rank blend:** average the two detectors' percentile ranks 50/50 and select exactly 6,531 alerts, testing whether complementary ordering helps at the original alert budget.

Security flags and stability proxies are **not** validation labels or F₃ scores. These variants are fixed before any new Kaggle score is seen.

In [3]:
raw_names={'ocsvm':'t5_r2_ocsvm_nu10_engineered_v2',
           'gmm':'t5_r2_gmm_engineered_v2'}
scores={}
for family,name in raw_names.items():
    model,_=saved_detector(name)
    values=np.asarray(model.anomaly_score(X),dtype=float)
    assert values.shape==(len(X),) and np.isfinite(values).all()
    scores[family]=values
rank_blend=.5*rankdata(scores['ocsvm'],method='average')+.5*rankdata(scores['gmm'],method='average')
selected=np.lexsort((np.arange(len(X)),-rank_blend))[:int(o25.sum())]
blend25=np.zeros(len(X),dtype=np.int8);blend25[selected]=1
variants={
    2:{'name':'kaggle_t12345_t3log_t5_ocsvm35_v14','prediction':o35,
       'hypothesis':'Broader OCSVM alert budget may improve recall enough to raise the Task 5 metric',
       'source':'t5_r5_ocsvm_nu10_alert35_v5'},
    3:{'name':'kaggle_t12345_t3log_t5_consensus_v15','prediction':(o25 & g25).astype(np.int8),
       'hypothesis':'OCSVM and GMM agreement may improve precision enough to offset fewer alerts',
       'source':'fixed intersection of submitted v9 OCSVM and v11 GMM predictions'},
    4:{'name':'kaggle_t12345_t3log_t5_union_v16','prediction':(o25 | g25).astype(np.int8),
       'hypothesis':'OCSVM and GMM union may improve recall enough to offset more false alerts',
       'source':'fixed union of submitted v9 OCSVM and v11 GMM predictions'},
    5:{'name':'kaggle_t12345_t3log_t5_rankblend25_v17','prediction':blend25,
       'hypothesis':'Equal percentile-rank blend may improve anomaly ordering at the same 6531-alert budget',
       'source':'fixed equal percentile-rank blend of saved OCSVM and GMM anomaly scores'},
}
for slot,spec in variants.items():
    p=spec['prediction']
    assert p.shape==(len(slot1),) and set(np.unique(p))=={0,1}
    print(slot,spec['name'],'alerts',int(p.sum()),'changed vs slot 1',int((p!=o25).sum()))
assert int((blend25!=o25).sum())>0 and blend25.sum()==o25.sum()

2 kaggle_t12345_t3log_t5_ocsvm35_v14 alerts 9118 changed vs slot 1 2587
3 kaggle_t12345_t3log_t5_consensus_v15 alerts 5377 changed vs slot 1 1154
4 kaggle_t12345_t3log_t5_union_v16 alerts 7685 changed vs slot 1 1154
5 kaggle_t12345_t3log_t5_rankblend25_v17 alerts 6531 changed vs slot 1 1084


## 3. Write four final CSVs and verify every column

Check all destination names before writing. Each card records source hashes, the fixed hypothesis, alert count, and changed rows. Never overwrite an existing submission artifact.

In [4]:
destinations={slot:(OUT/f"{spec['name']}.csv",OUT/f"{spec['name']}.json")
              for slot,spec in variants.items()}
existing=[(slot,csv.exists(),meta.exists()) for slot,(csv,meta) in destinations.items()]
assert all(a==b for _,a,b in existing),'Inspect incomplete destination artifacts'
assert all(a for _,a,_ in existing) or not any(a for _,a,_ in existing),       'Inspect mixed existing and missing destination artifacts'
outputs={}
for slot,spec in variants.items():
    output=slot1.copy();output['task5']=spec['prediction'].astype(int)
    assert output.columns.tolist()==columns and output.id.tolist()==sample.id.tolist()
    assert output[['id','task1','task2','task3','task4']].equals(slot1[['id','task1','task2','task3','task4']])
    assert output.notna().all().all()
    outputs[slot]=output
if not any(a for _,a,_ in existing):
    for slot,spec in variants.items():
        csv,meta=destinations[slot]
        output=outputs[slot]
        output.to_csv(csv,index=False)
        detail={'name':spec['name'],'date':'2026-11-14','slot':slot,
            'source_submission':'submission/'+slot1_path.name,
            'source_sha256':sha(slot1_path),'changed_task':'task5',
            'hypothesis':spec['hypothesis'],'task5_source':spec['source'],
            'task5_orientation':'1 = anomaly','task5_alerts':int(spec['prediction'].sum()),
            'changed_task5_rows_vs_slot1':int((spec['prediction']!=o25).sum()),
            'rows':len(output),'columns':columns,'seed':SEED,
            'source_task5_csv_sha256':{'v9_ocsvm25':sha(v9_path),'v10_ocsvm35':sha(v10_path),
                                      'v11_gmm25':sha(v11_path)},
            'score_status':'unsubmitted','output_sha256':sha(csv)}
        meta.write_text(json.dumps(detail,indent=2)+'\n',encoding='utf-8')
for slot,spec in variants.items():
    csv,meta=destinations[slot]
    saved=pd.read_csv(csv,dtype=str);card=json.loads(meta.read_text(encoding='utf-8'))
    assert card['output_sha256']==sha(csv) and card['source_sha256']==sha(slot1_path)
    assert saved.columns.tolist()==columns and saved.id.tolist()==sample.id.tolist()
    assert saved[['id','task1','task2','task3','task4']].equals(slot1[['id','task1','task2','task3','task4']])
    assert np.array_equal(saved.task5.astype(int),spec['prediction'])
    print('Verified slot',slot,csv.name,'SHA-256',card['output_sha256'])

Verified slot 2 kaggle_t12345_t3log_t5_ocsvm35_v14.csv SHA-256 a549968b215f71c00fc89c9ce1f366a7b0a1295aaa688bf56774ad40daaf9bd3
Verified slot 3 kaggle_t12345_t3log_t5_consensus_v15.csv SHA-256 e243d344b2c21f4fd8b1c0b87d690dea91fd8237b47629544b69628d251c3662
Verified slot 4 kaggle_t12345_t3log_t5_union_v16.csv SHA-256 f901f69b11b5cb065d1c20d9d1e67b543f1b389486fa00aeb279dcf2680fdf16
Verified slot 5 kaggle_t12345_t3log_t5_rankblend25_v17.csv SHA-256 d4b7c7b32c8c051a3d398819523ddfde4373699a7c2c2537b30ed912b519a9c8
